<a href="https://colab.research.google.com/github/arya2596/LinearRegression/blob/main/Neural_Project_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Bengaluru_House_Data_Neural.xlsx to Bengaluru_House_Data_Neural.xlsx


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow import keras
from tensorflow.keras import layers

# Load dataset
df = pd.read_excel('Bengaluru_House_Data_Neural.xlsx', header=1)

# Clean total_sqft column (handle ranges like "2100-2850" or "15Acres")
def convert_sqft(x):
    try:
        if '-' in str(x):
            vals = x.split('-')
            return (float(vals[0]) + float(vals[1])) / 2
        return float(''.join([c for c in str(x) if c.isdigit() or c == '.']))
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df.dropna(subset=['location', 'total_sqft', 'bath', 'price'], inplace=True)

# Extract numeric BHK value
df['bhk'] = df['size'].apply(lambda x: int(str(x).split(' ')[0]) if isinstance(x, str) else np.nan)
df.dropna(subset=['bhk'], inplace=True)

# Encode location
le = LabelEncoder()
df['location'] = le.fit_transform(df['location'])

# Select features and label
X = df[['location', 'bhk', 'total_sqft', 'bath']]
y = df['price']

# Train-test split and scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data cleaned and ready for model training!")


✅ Data cleaned and ready for model training!


In [3]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mse'])

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1
)


Epoch 1/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 33015.2969 - mse: 33015.2969 - val_loss: 17073.3242 - val_mse: 17073.3242
Epoch 2/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21689.1074 - mse: 21689.1074 - val_loss: 14834.8701 - val_mse: 14834.8701
Epoch 3/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 12813.3154 - mse: 12813.3154 - val_loss: 14216.3477 - val_mse: 14216.3477
Epoch 4/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14548.4844 - mse: 14548.4844 - val_loss: 14100.8750 - val_mse: 14100.8750
Epoch 5/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 14227.1143 - mse: 14227.1143 - val_loss: 13848.0488 - val_mse: 13848.0488
Epoch 6/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 11989.5439 - mse: 11989.5439 - val_loss: 13707.5127 - val_mse: 13707.5127
Epoch 7/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13086.2070 - mse: 13086.2070 - val_loss: 13603.8076 - val_mse: 13603.8076
Epoch 8/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - 

In [4]:
print("\nEnter house details to predict price:")

sqft = float(input("Total square feet: "))
bath = float(input("Number of bathrooms: "))
bhk = float(input("Number of bedrooms (BHK): "))
loc = input("Location: ").title()

# Convert location safely
if loc in le.classes_:
    loc_val = le.transform([loc])[0]
else:
    print("⚠️ Location not found, using default value 0")
    loc_val = 0

user_data = np.array([[loc_val, bhk, sqft, bath]])
user_scaled = scaler.transform(user_data)

pred = model.predict(user_scaled)[0][0]
print(f"\n🏠 Estimated Price: ₹{pred:,.2f} lakhs (approx.)")



Enter house details to predict price:
Total square feet: 14566
Number of bathrooms: 6
Number of bedrooms (BHK): 6
Location: koramangala
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

🏠 Estimated Price: ₹1,306.69 lakhs (approx.)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [5]:
from sklearn.metrics import r2_score
y_pred = model.predict(X_test_scaled).flatten()
r2 = r2_score(y_test, y_pred)*100
print(f"accuracy is {r2}%")


83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 826us/step
accuracy is 52.68238336150368%
